# Código de proceso ETL

In [1]:
import pandas as pd
import psycopg2
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv
from sqlalchemy.pool import NullPool

# 1. Conexión Base de Datos

In [2]:
# Load environment variables from .env
load_dotenv(dotenv_path=".env", override=True)

USER = os.getenv("user")
PASSWORD = os.getenv("password")
HOST = os.getenv("host")
PORT = os.getenv("port")
DBNAME = os.getenv("dbname")

print("USER:", USER)
print("HOST:", HOST)
print("PORT:", PORT)

DATABASE_URL = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}?sslmode=require"

engine = create_engine(
    DATABASE_URL,
    poolclass=NullPool
)

try:
    with engine.connect() as connection:
        print("Connection successful!")
except Exception as e:
    print(f"Failed to connect: {e}")

USER: postgres.belcfffqjrnqrcqavsxz
HOST: aws-1-us-east-2.pooler.supabase.com
PORT: 6543
Connection successful!


# 2. Extracción de datos

In [3]:
csv_file = "customer_shopping_data.csv"
df = pd.read_csv(csv_file)
df.head()


,invoice_no,customer_id,gender,age,category,quantity,price,payment_method,invoice_date,shopping_mall
0,I138884,C241288,Female,28,Clothing,5,1500.40,Credit Card,5/8/2022,Kanyon
1,I317333,C111565,Male,21,Shoes,3,1800.51,Debit Card,12/12/2021,Forum Istanbul
2,I127801,C266599,Male,20,Clothing,1,300.08,Cash,9/11/2021,Metrocity
3,I173702,C988172,Female,66,Shoes,5,3000.85,Credit Card,16/05/2021,Metropol AVM
4,I337046,C189076,Female,53,Books,4,60.60,Cash,24/10/2021,Kanyon


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99457 entries, 0 to 99456
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   invoice_no      99457 non-null  str    
 1   customer_id     99457 non-null  str    
 2   gender          99457 non-null  str    
 3   age             99457 non-null  int64  
 4   category        99457 non-null  str    
 5   quantity        99457 non-null  int64  
 6   price           99457 non-null  float64
 7   payment_method  99457 non-null  str    
 8   invoice_date    99457 non-null  str    
 9   shopping_mall   99457 non-null  str    
dtypes: float64(1), int64(2), str(7)
memory usage: 7.6 MB


# 3. Trasformación

In [5]:
"""
Covertir la letra inicial de los identificadores en un número espécifico 
para mantener esa diferenciación 
y convertirlos en tipo enteros
"""
df['invoice_no'] = df['invoice_no'].replace("^I", "1", regex=True).astype(int)
df['customer_id'] = df['customer_id'].replace("^C", "1", regex=True).astype(int)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99457 entries, 0 to 99456
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   invoice_no      99457 non-null  int64  
 1   customer_id     99457 non-null  int64  
 2   gender          99457 non-null  str    
 3   age             99457 non-null  int64  
 4   category        99457 non-null  str    
 5   quantity        99457 non-null  int64  
 6   price           99457 non-null  float64
 7   payment_method  99457 non-null  str    
 8   invoice_date    99457 non-null  str    
 9   shopping_mall   99457 non-null  str    
dtypes: float64(1), int64(4), str(5)
memory usage: 7.6 MB


In [6]:
#Se convierte la fecha de compra en un tipo fecha
df['invoice_date'] = pd.to_datetime(df['invoice_date'], format="%d/%m/%Y")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99457 entries, 0 to 99456
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   invoice_no      99457 non-null  int64         
 1   customer_id     99457 non-null  int64         
 2   gender          99457 non-null  str           
 3   age             99457 non-null  int64         
 4   category        99457 non-null  str           
 5   quantity        99457 non-null  int64         
 6   price           99457 non-null  float64       
 7   payment_method  99457 non-null  str           
 8   invoice_date    99457 non-null  datetime64[us]
 9   shopping_mall   99457 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(4), str(4)
memory usage: 7.6 MB


In [7]:
df.rename(columns={'customer_id': 'id_customer'}, inplace=True)
df.rename(columns={'invoice_no': 'id_invoice'}, inplace=True)
df.head()

,id_invoice,id_customer,gender,age,category,quantity,price,payment_method,invoice_date,shopping_mall
0,1138884,1241288,Female,28,Clothing,5,1500.40,Credit Card,2022-08-05,Kanyon
1,1317333,1111565,Male,21,Shoes,3,1800.51,Debit Card,2021-12-12,Forum Istanbul
2,1127801,1266599,Male,20,Clothing,1,300.08,Cash,2021-11-09,Metrocity
3,1173702,1988172,Female,66,Shoes,5,3000.85,Credit Card,2021-05-16,Metropol AVM
4,1337046,1189076,Female,53,Books,4,60.60,Cash,2021-10-24,Kanyon


### Creamos la tabla dim_customer

In [8]:
dim_customer = df[["id_customer", "gender", "age"]].copy()
dim_customer.head()

,id_customer,gender,age
0,1241288,Female,28
1,1111565,Male,21
2,1266599,Male,20
3,1988172,Female,66
4,1189076,Female,53


### Ahora creamos la tabla dim_category

In [9]:
dim_category = df[["category"]].copy()
dim_category["id_category"] = dim_category.index + 1
dim_category.head()

,category,id_category
0,Clothing,1
1,Shoes,2
2,Clothing,3
3,Shoes,4
4,Books,5


### Creamos la tabla dim_date

In [10]:
dim_date = df[["invoice_date"]].copy()
dim_date["id_date"] = dim_category.index + 1
dim_date["day"] = dim_date["invoice_date"].dt.day
dim_date["month"] = dim_date["invoice_date"].dt.month
dim_date["year"] = dim_date["invoice_date"].dt.year
dim_date.head()

,invoice_date,id_date,day,month,year
0,2022-08-05,1,5,8,2022
1,2021-12-12,2,12,12,2021
2,2021-11-09,3,9,11,2021
3,2021-05-16,4,16,5,2021
4,2021-10-24,5,24,10,2021


### Creamos la tabla dim_store

In [11]:
dim_store = df[["shopping_mall"]].copy()
dim_store["id_mall"] = dim_store.index + 1
dim_store.head()

,shopping_mall,id_mall
0,Kanyon,1
1,Forum Istanbul,2
2,Metrocity,3
3,Metropol AVM,4
4,Kanyon,5


### Creamos la tabla dim_payment

In [12]:
dim_payment = df[["payment_method"]].copy()
dim_payment["id_payment_method"] = dim_payment.index + 1
dim_payment.head()

,payment_method,id_payment_method
0,Credit Card,1
1,Debit Card,2
2,Cash,3
3,Credit Card,4
4,Cash,5


### Creamos la tabla de hechos

In [22]:
facts_sales = df[["price", "quantity", "id_invoice"]].copy()
facts_sales["id_category"] = dim_category["id_category"]
facts_sales["id_date"] = dim_date["id_date"]
facts_sales['id_mall'] = dim_store["id_mall"] 
facts_sales['id_payment_method'] = dim_payment["id_payment_method"] 
facts_sales.head()

,price,quantity,id_invoice,id_category,id_date,id_mall,id_payment_method
0,1500.40,5,1138884,1,1,1,1
1,1800.51,3,1317333,2,2,2,2
2,300.08,1,1127801,3,3,3,3
3,3000.85,5,1173702,4,4,4,4
4,60.60,4,1337046,5,5,5,5


## Enviamos las tablas a la base de datos

In [28]:
dim_customer.to_sql("dim_customer", engine, index=False, if_exists="replace")


InternalError: (psycopg2.errors.DependentObjectsStillExist) cannot drop table dim_customer because other objects depend on it
DETAIL:  constraint fact_sales_id_customer_fkey on table fact_sales depends on table dim_customer
HINT:  Use DROP ... CASCADE to drop the dependent objects too.

[SQL: 
DROP TABLE dim_customer]
(Background on this error at: https://sqlalche.me/e/20/2j85)